![Snowflake](https://www.snowflake.com/wp-content/themes/flavor/assets/img/logo-snowflake-sans-word.svg)
# CredibanCo — Hands-On Lab
**RFP 10010806** · Plataforma de Datos · Snowflake 2026

---

# Track 7 — Administración, DataOps, FinOps y Salud de Plataforma
**Rol:** CRB_INFRAESTRUCTURA | **Tiempo:** 15 min | **Audiencia:** Equipo de infraestructura y finanzas

En este track exploramos las capacidades de **FinOps y gobierno de costos** de Snowflake. La plataforma incluye dashboards nativos de gestión de costos que no requieren código ni configuración adicional — están listos para usar desde el primer día.

In [ ]:
-- Configurar el contexto
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## Bloque 1 — Cost Management: Dashboard Nativo

Snowflake incluye un **dashboard de costos** completo, listo para usar. No es un reporte que hay que construir — es parte de la plataforma.

### 👁️ Navegar a Cost Management

1. En el menú izquierdo de Snowsight, ir a **Admin > Cost Management**
2. Seleccionar el warehouse `CREDIBANCO_HOL_WH` si lo pide
3. Ir a la pestaña **Account Overview**

### Qué van a ver:

| Sección | Qué muestra |
|---------|-------------|
| **Cost Summary** | Gasto total del período, % cambio vs período anterior, promedio diario |
| **Monthly Budget** | % del presupuesto consumido, estado on-track/over |
| **Anomalies** | Días con consumo fuera del rango esperado (detección automática) |
| **Optimization Insights** | Recomendaciones concretas de ahorro con créditos estimados |
| **Total Cost (gráfico)** | Tendencia de gasto por día/semana/mes con desglose por servicio |
| **Warehouse Cost Attribution** | Chargeback: consumo por tag de dominio (equipo/área) |
| **Top Spend** | Top warehouses, queries, databases por costo |

Todo esto se genera **automáticamente** — sin queries, sin dashboards externos, sin configuración.

## Bloque 2 — Consumption: Desglose detallado

### 👁️ Navegar a Consumption

1. En **Admin > Cost Management**, ir a la pestaña **Consumption**
2. En el filtro **All Usage Types**, seleccionar **Compute** para ver solo warehouses
3. Cambiar el rango de fechas para ver el último mes

### Qué van a ver:
- Gráfico de barras con costo diario por tipo de servicio
- Desglose: Compute, Storage, Data Transfer, AI Services
- Filtros por tipo, warehouse, fecha

**Punto clave:** En Snowflake el costo se desglosa automáticamente por servicio. No hay costos ocultos en "overhead" de cluster como en Databricks — cada crédito se atribuye a un servicio específico.

## Bloque 3 — Budgets: Presupuestos y Alertas

### 👁️ Navegar a Budgets

1. En **Admin > Cost Management**, ir a la pestaña **Budgets**
2. Ver el presupuesto de la cuenta

Los **Budgets** permiten:
- Establecer un **límite mensual** de créditos por cuenta o por grupo de objetos
- Recibir **alertas automáticas** cuando el gasto proyectado supera el límite
- Crear **presupuestos personalizados** por equipo, proyecto o ambiente

Ahora creamos un presupuesto desde SQL:

In [ ]:
-- Crear un presupuesto custom para el warehouse del HOL
CALL SNOWFLAKE.LOCAL.ACCOUNT_ROOT_BUDGET!ACTIVATE();

-- Ver el estado del presupuesto de la cuenta
CALL SNOWFLAKE.LOCAL.ACCOUNT_ROOT_BUDGET!GET_SPENDING_HISTORY(
  TIME_LOWER_BOUND => DATEADD('days', -30, CURRENT_TIMESTAMP()),
  TIME_UPPER_BOUND => CURRENT_TIMESTAMP()
);

## Bloque 4 — Resource Monitors: Control de Gasto

Los **Resource Monitors** son la segunda línea de defensa: controlan cuántos créditos puede consumir un warehouse y toman acción automática (notificar, suspender) al alcanzar umbrales.

In [ ]:
-- Crear Resource Monitor con alertas escalonadas
CREATE OR REPLACE RESOURCE MONITOR MONITOR_HOL
  WITH CREDIT_QUOTA = 100
  FREQUENCY = MONTHLY
  START_TIMESTAMP = IMMEDIATELY
  TRIGGERS
    ON 75 PERCENT DO NOTIFY
    ON 90 PERCENT DO NOTIFY
    ON 100 PERCENT DO SUSPEND;

### 👁️ Ver Resource Monitors en la UI

1. Ir a **Admin > Resource Monitors**
2. Ver `MONITOR_HOL` con su configuración de umbrales
3. Observar cómo los triggers están configurados: notificar al 75%, al 90% y suspender al 100%

Esto garantiza que **ningún equipo puede exceder su presupuesto** sin autorización explícita.

## Bloque 5 — Detección de Anomalías

### 👁️ Volver a Account Overview

1. En **Admin > Cost Management > Account Overview**
2. Buscar la sección **Anomalies**
3. Si hay anomalías detectadas, hacer click en cualquiera para ver el detalle

Snowflake usa un **modelo estadístico automático** que:
- Calcula el rango esperado de consumo diario
- Marca como anomalía cualquier día fuera del rango
- Muestra la desviación exacta (Over/Under expected)
- Permite investigar con un click: se abre Cortex Code con el contexto de la anomalía

No hay que configurar nada — la detección de anomalías viene incluida.

## Bloque 6 — Cortex Code: Preguntas ad-hoc sobre costos

Desde el dashboard de Cost Management, cada sección tiene un botón **+** que abre Cortex Code con contexto preloaded. También puede hacer preguntas libres:

### 🤖 Prompt para CoCo:

> **¿Cuáles son las 10 queries más costosas de la última semana? Muestra el usuario, el warehouse, los créditos consumidos y la duración. Ordena por costo descendente.**

> **Crea un dashboard Streamlit de FinOps que muestre: KPI cards con créditos totales y promedio diario, gráfico de consumo por día, distribución por servicio, y tabla de top warehouses. Usa los colores de Snowflake.**

Estos prompts complementan el dashboard nativo cuando se necesita un análisis personalizado o una visualización específica.

---
## Verificación

| Capacidad | Cómo se validó |
|-----------|---------------|
| Visibilidad de costos | Dashboard nativo en Admin > Cost Management > Account Overview |
| Desglose por servicio | Pestaña Consumption con filtros por tipo y warehouse |
| Presupuestos | Budget activado + Resource Monitor con umbrales |
| Anomalías | Detección automática en Account Overview > Anomalies |
| Chargeback | Warehouse Cost Attribution por tags de dominio |
| Optimización | Optimization Insights con recomendaciones y ahorro estimado |
| Análisis ad-hoc | Cortex Code para queries personalizadas y dashboards Streamlit |

**La plataforma incluye FinOps nativo** — no requiere herramientas externas, configuración adicional ni código. Los equipos de finanzas e infraestructura de CredibanCo pueden operar desde el día 1.